# Training of ML solution

#### [Applying machine learning on sensor data for irrigation recommendations: revealing the agronomist’s tacit knowledge](https://link.springer.com/article/10.1007/s11119-017-9527-4), [Precision Agriculture](https://link.springer.com/journal/11119), 2018

In [194]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from math import log
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import make_scorer, mean_squared_error

In [195]:
weather_file = os.path.join("/", "home", "data", "ml_data", "raw_data", "weather_1day.csv")
irrigation_groundtruth_file = os.path.join("/", "home", "data", "ml_data", "raw_data", "irrigation_1day.csv")
soil_moisture_file = os.path.join("/", "home", "data", "ml_data", "raw_data", "soil_moisture_1day.csv")

weather_dataset = pd.read_csv(weather_file)
irrigation_dataset = pd.read_csv(irrigation_groundtruth_file)
soil_moisture_dataset = pd.read_csv(soil_moisture_file)

#### Preprocessing data

In [196]:
# Removing useless meteo variables
weather_dataset = weather_dataset.loc[weather_dataset["detectedValueTypeId"].isin(["AIR_HUM","AIR_TEMP","SOLAR_RAD",])]
weather_dataset["value"] = weather_dataset["value"].apply(lambda row: round(row,2))
# Pivoting weather data
weather_dataset = weather_dataset.pivot_table(
    index=["date", "fieldName", "sectorName"],
    columns="detectedValueTypeId",
    values="value"
).reset_index()
weather_dataset.dropna(subset=["AIR_HUM"], inplace=True)


soil_moisture_dataset["yy"] = soil_moisture_dataset["yy"].apply(lambda x : abs(x))
soil_moisture_dataset["value"] = soil_moisture_dataset["value"].apply(lambda row: round(row,2))
# Pivoting moisture data
soil_moisture_dataset = soil_moisture_dataset.pivot_table(
    index=["date", "fieldName", "sectorName"],
    columns="yy",
    values="value"
).reset_index()

soil_moisture_dataset = soil_moisture_dataset.rename(columns={20 : 'moistureDepth20cm', 60 : 'moistureDepth60cm'})
soil_moisture_dataset.dropna(subset=["moistureDepth60cm"], inplace=True)
soil_moisture_dataset.dropna(subset=["moistureDepth20cm"], inplace=True)

# Unifying field name across years
weather_dataset["fieldName"] = "Fondo Errano"
irrigation_dataset["fieldName"] = "Fondo Errano"
soil_moisture_dataset["fieldName"] = "Fondo Errano"

# Create dataset
dataset = soil_moisture_dataset.merge(weather_dataset, on = ["date","fieldName","sectorName"], how = 'inner').merge(irrigation_dataset,on = ["date","fieldName","sectorName"], how = 'inner')
# Removing useless categorical attributes
dataset = dataset.drop(columns=["date","fieldName","sectorName"])
print(f"Dataset has now {len(dataset)} rows with \n{dataset.isna().sum()}\n null values")

Dataset has now 1272 rows with 
moistureDepth20cm    0
moistureDepth60cm    0
AIR_HUM              0
AIR_TEMP             0
SOLAR_RAD            0
irrigation           0
dtype: int64
 null values


#### Add forecasts, IPPD, saturation and drought data

In [197]:
meteo_cols = ["AIR_HUM", "AIR_TEMP", "SOLAR_RAD"]
window = 1 # days
for col in meteo_cols:
    dataset[f"{col}_forecast"] = dataset[col].shift(-1)
    dataset.loc[dataset.index[-1], f"{col}_forecast"] = dataset.loc[dataset.index[-1], col]

    dataset[f"{col}_IPPD"] = dataset[col].shift(1)
    dataset.loc[dataset.index[0], f"{col}_IPPD"] = dataset.loc[dataset.index[0], col]

for depth in [20, 60]:
    col = f"moistureDepth{depth}cm"

    dataset[f"drought_duration_{depth}cm"] = (
        (dataset[col] <= -300)
        .astype(int)
        .rolling(window=window, min_periods=1)
        .sum()
    )

    # Saturation duration negli ultimi 7 giorni
    dataset[f"saturation_duration_{depth}cm"] = (
        (dataset[col] >= -50)
        .astype(int)
        .rolling(window=window, min_periods=1)
        .sum()
    )

dataset

,moistureDepth20cm,moistureDepth60cm,AIR_HUM,AIR_TEMP,SOLAR_RAD,irrigation,AIR_HUM_forecast,AIR_HUM_IPPD,AIR_TEMP_forecast,AIR_TEMP_IPPD,SOLAR_RAD_forecast,SOLAR_RAD_IPPD,drought_duration_20cm,saturation_duration_20cm,drought_duration_60cm,saturation_duration_60cm
0,-20.20,-24.03,67.44,13.00,15.99,0.0,66.65,67.44,12.95,13.00,59.59,15.99,0.0,1.0,0.0,1.0
1,-21.00,-23.90,66.65,12.95,59.59,0.0,41.03,67.44,23.59,13.00,231.43,15.99,0.0,1.0,0.0,1.0
2,-17.91,-23.42,41.03,23.59,231.43,0.0,40.14,66.65,20.26,12.95,238.17,59.59,0.0,1.0,0.0,1.0
3,-17.48,-23.17,40.14,20.26,238.17,0.0,60.13,41.03,14.39,23.59,104.87,231.43,0.0,1.0,0.0,1.0
4,-16.96,-22.89,60.13,14.39,104.87,0.0,68.19,40.14,7.95,20.26,231.23,238.17,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1267,-26.59,-340.11,71.36,17.34,159.23,0.0,72.53,81.92,16.64,15.23,141.74,86.73,0.0,1.0,1.0,0.0
1268,-28.35,-334.94,72.53,16.64,141.74,0.0,84.21,71.36,14.36,17.34,57.48,159.23,0.0,1.0,1.0,0.0
1269,-28.87,-458.39,84.21,14.36,57.48,0.0,71.45,72.53,11.88,16.64,146.04,141.74,0.0,1.0,1.0,0.0
1270,-24.84,-492.55,71.45,11.88,146.04,0.0,80.63,84.21,9.63,14.36,28.56,57.48,0.0,1.0,1.0,0.0


## Utility function: change dataset time granularity

In [198]:
def change_time_granularity(df: pd.DataFrame, days: int) -> pd.DataFrame:
    data = df.copy()
    data["date"] = pd.to_datetime(data["date"])
    data = data.set_index("date")

    agg_dict = {
        "moistureDepth20cm": "mean",
        "moistureDepth60cm": "mean",
        "AIR_HUM": "mean",
        "AIR_TEMP": "mean",
        "SOLAR_RAD": "mean",
        "irrigation": "sum",
        "AIR_HUM_forecast": "mean",
        "AIR_HUM_IPPD": "mean",
        "AIR_TEMP_forecast": "mean",
        "AIR_TEMP_IPPD": "mean",
        "SOLAR_RAD_forecast": "mean",
        "SOLAR_RAD_IPPD": "mean",
        "drought_duration_20cm": "sum",
        "saturation_duration_20cm": "sum",
        "drought_duration_60cm": "sum",
        "saturation_duration_60cm": "sum"
    }


    df_resampled = data.resample(f"{days}D").agg(agg_dict).reset_index()

    return df_resampled

### Model preparation

In [199]:
y = dataset["irrigation"]  # labels
X = dataset.drop(columns=["irrigation"])  # input data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

X_train: (1017, 15)
X_test: (255, 15)
y_train: (1017,)
y_test: (255,)


#### Hyperparameter optimization through AutoML - Gradient Boost Regression Trees

In [ ]:
gbr = GradientBoostingRegressor(random_state=42)

param_dist = {
    "n_estimators": np.arange(100, 1001, 100),    # numero di alberi
    "learning_rate": np.linspace(0.01, 0.2, 20),  # tasso di apprendimento
    "max_depth": np.arange(2, 8),                 # profondità massima degli alberi
    "subsample": np.linspace(0.6, 1.0, 5),        # frazione di campioni per albero
    "min_samples_split": np.arange(2, 11),        # minimo campioni per split
    "min_samples_leaf": np.arange(1, 11)          # minimo campioni in foglia
}

rmse_scorer = make_scorer(lambda y_true, y_pred: -np.sqrt(mean_squared_error(y_true, y_pred)))

random_search = RandomizedSearchCV(
    estimator=gbr,
    param_distributions=param_dist,
    n_iter=600,              # numero di combinazioni da provare
    scoring=rmse_scorer,
    cv=3,                   # cross-validation a 3 fold
    verbose=2,
    random_state=42,
    n_jobs=-1               # usa tutti i core disponibili
)

random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)

best_gbr = random_search.best_estimator_

y_pred = best_gbr.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")

Fitting 3 folds for each of 600 candidates, totalling 1800 fits


[CV] END learning_rate=0.09444444444444444, max_depth=5, min_samples_leaf=2, min_samples_split=2, n_estimators=200, subsample=0.9; total time=   1.3s
[CV] END learning_rate=0.09444444444444444, max_depth=5, min_samples_leaf=2, min_samples_split=2, n_estimators=200, subsample=0.9; total time=   1.4s
[CV] END learning_rate=0.09444444444444444, max_depth=2, min_samples_leaf=6, min_samples_split=2, n_estimators=400, subsample=0.9; total time=   1.5s
[CV] END learning_rate=0.09444444444444444, max_depth=5, min_samples_leaf=2, min_samples_split=2, n_estimators=200, subsample=0.9; total time=   1.6s
[CV] END learning_rate=0.09444444444444444, max_depth=2, min_samples_leaf=6, min_samples_split=2, n_estimators=400, subsample=0.9; total time=   1.6s
[CV] END learning_rate=0.09444444444444444, max_depth=2, min_samples_leaf=6, min_samples_split=2, n_estimators=400, subsample=0.9; total time=   1.7s
[CV] END learning_rate=0.11555555555555555, max_depth=4, min_samples_leaf=7, min_samples_split=5, n_